In [1]:
import sys
import os
from pathlib import Path

# Add the project root path to sys.path
notebook_path = Path.cwd()
ROOT = notebook_path.parent 

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root added to sys.path: {ROOT}")

Project root added to sys.path: /srv/homes/onbo10/thesis_main


+ Select the GPU

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

* load general evaluation constants

In [3]:
from src.evaluation.constants import SIGMAS, IOU_THRESHOLDS

* imports

In [4]:
from src.utils  import *
import torch
from torch.utils.data import DataLoader
from src.evaluation.eval_datasets import HRNetEvaluationDataset
import numpy as np
import yaml
from types import SimpleNamespace
from hrnet_config import cfg , update_config

+ Evaluate HRNet (Cropped one instance input)

In [5]:

# Load finetuned HRNet
cfg_file = '../configs/HRNet/w32_256x192_adam_lr1e-3__cropped_out7-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/HRNet_one_instance/Experiment1/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
model= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
device= get_device()
model.to(device)
model.eval()


PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [8]:
# Load the test dataset and initiate the dataloader
img_root= '../data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_frames'
ann_root= '../data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_bboxes_kpts'
split_path='../data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']
dataset = HRNetEvaluationDataset(img_root,ann_root, mode='cropped',video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=16,shuffle=False)

=> In this dataset class we are skipping instances where the GT bounding boxes are corrupt (H or W are < 20 px or the area < 400 , or GT kpts not in bbox)

In [9]:
from src.evaluation.evaluation_utils import evaluate_cropped_HRNET

num_images, precision, recall, map50, map50_95 = evaluate_cropped_HRNET(model, test_loader,device ,SIGMAS, IOU_THRESHOLDS)
print("\n" + "="*40)
print("HRNET TEST EVALUATION RESULTS")
print("="*40)
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


HRNET TEST EVALUATION RESULTS
Total Images: 3073
Precision:    0.9997
Recall:       0.9997
mAP@50:       0.9950
mAP@50-95:    0.9401


* Evaluate YOLOv8x-pose

In [5]:
# In order to evaluate YOLO on the exact same instances of HRNET evaluation we need to detect relevent keys
img_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_frames'
ann_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_bboxes_kpts'
split_path='/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']
hrnet_dataset = SurgPoseDatasetOneInstanceInference(img_root, ann_root,video_list=test_vid_list )
valid_keys = set()

for sample in hrnet_dataset.samples:
    # Create a unique key: 'video_id/left or right'
    img_name = os.path.basename(sample["img_path"])
    video_id = sample["img_path"].split('/')[-2]
    key = f"{video_id}/{img_name}_{sample['obj_id']}"
    valid_keys.add(key)

print(f"Total instances in HRNet evaluation: {len(valid_keys)}")

[SurgPoseDataset] Kept 3073 / 4008 samples  (skipped 935)
Total instances in HRNet evaluation: 3073


In [5]:
from evaluation_dataset import YoloPoseDataset
from ultralytics import YOLO

device= get_device()
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)
SIGMAS_YOLO= torch.tensor([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079]).to(device) #np.ones(7) / 7
yolo_model = YOLO('/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/Yolo/runs/pose/yolov8pose_surgpose2/weights/last.pt')
dataset = YoloPoseDataset(img_dir='/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/images/test', label_dir='/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/labels/test')
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)


In [8]:
from evaluation_utils import evaluate_YOLO

num_images, precision, recall, map50, map50_95, num_valid = evaluate_YOLO(yolo_model, dataloader, device,SIGMAS_YOLO, IOU_THRESHOLDS, valid_keys, ann_root)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)



/srv/homes/onbo10/.conda/envs/HRNetEnv_v2/lib/python3.10/site-packages/ultralytics/utils/metrics.py:184: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = torch.tensor(sigma, device=kpt1.device, dtype=kpt1.dtype)  # (17, )



YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.8248
Recall:       0.9906
mAP@50:       0.8616
mAP@50-95:    0.6322


* Use different SIGMAS for evaluation (OKS computation)

In [9]:
device= get_device()
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)
SIGMAS_uniform= np.ones(7) / 7 

In [13]:
# HRNET
from evaluation_utils import evaluate_cropped_HRNET

num_images, precision, recall, map50, map50_95 = evaluate_cropped_HRNET(model, test_loader,device ,SIGMAS_uniform, IOU_THRESHOLDS)
print("\n" + "="*40)
print("HRNET TEST EVALUATION RESULTS")
print("="*40)
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


HRNET TEST EVALUATION RESULTS
Total Images: 3073
Precision:    1.0000
Recall:       1.0000
mAP@50:       0.9950
mAP@50-95:    0.9950


In [15]:
# YOLO
from evaluation_utils import evaluate_YOLO

num_images, precision, recall, map50, map50_95, num_valid = evaluate_YOLO(yolo_model, dataloader, device,SIGMAS_uniform, IOU_THRESHOLDS, valid_keys, ann_root)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)



YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.8251
Recall:       0.9912
mAP@50:       0.8622
mAP@50-95:    0.8612


* HRNET full image 256*192 with data augmentation

In [6]:
# Load finetuned HRNet
cfg_file = '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/Experiment4/training_chekpoints2026-01-09_17-09-50/model_epoch200.pth'

hrnet_model= load_pretrained_HRNet(cfg_file,model_weights, finetuned=True)
device= get_device()
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [8]:
# Load the test dataset and initiate the dataloader
from evaluation_dataset import SurgPoseInferenceDataset
img_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_frames'
ann_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_bboxes_kpts'
split_path='/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']
dataset = SurgPoseInferenceDataset(img_root,ann_root,input_size=(256, 192), heatmap_size=(64, 48), video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [9]:
# Define constants for metric computation
SIGMAS =  torch.tensor([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079]).to(device) #np.ones(7) / 7
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)

In [10]:
from evaluation_utils import evaluate_HRNet_full_image

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root, w_m=64.0, h_m=48.0)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)

/srv/homes/onbo10/thesis_Ons/Evaluation/evaluation_utils.py:282: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  gt_kpts = torch.tensor(gt_kpts_valid, device=device, dtype=torch.float32)
/srv/homes/onbo10/.conda/envs/HRNetEnv_v2/lib/python3.10/site-packages/ultralytics/utils/metrics.py:184: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = torch.tensor(sigma, device=kpt1.device, dtype=kpt1.dtype)  # (17, )



YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.9281
Recall:       0.9388
mAP@50:       0.9228
mAP@50-95:    0.4053


* HRNET full image 256*192 without data augmentation

In [11]:
# Load finetuned HRNet
cfg_file = '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/Experiment4/training_chekpoints2026-01-12_09-18-19/model_epoch200.pth'

hrnet_model= load_pretrained_HRNet(cfg_file,model_weights, finetuned=True)
device= get_device()
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [12]:
# Load the test dataset and initiate the dataloader
from evaluation_dataset import SurgPoseInferenceDataset
img_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_frames'
ann_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_bboxes_kpts'
split_path='/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']
dataset = SurgPoseInferenceDataset(img_root,ann_root,input_size=(256, 192), heatmap_size=(64, 48), video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [13]:
# Define constants for metric computation
SIGMAS =  torch.tensor([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079]).to(device) #np.ones(7) / 7
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)

In [14]:
from evaluation_utils import evaluate_HRNet_full_image

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root, w_m=64.0, h_m=48.0)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.8929
Recall:       0.9229
mAP@50:       0.9016
mAP@50-95:    0.3856


* HRNET full image 704*512 with data augmentation

In [15]:
# Load finetuned HRNet
cfg_file = '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/w32_704x512_adam_lr1e-3_out14-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet-experiments/HRNet_finetuned/Experiment4/training_chekpoints2026-01-09_17-08-56/model_epoch200.pth'

hrnet_model= load_pretrained_HRNet(cfg_file,model_weights, finetuned=True)
device= get_device()
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [17]:
# Load the test dataset and initiate the dataloader
from evaluation_dataset import SurgPoseInferenceDataset
img_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_frames'
ann_root= '/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/extracted_bboxes_kpts'
split_path='/srv/homes/onbo10/thesis_Ons/SurgePoseData/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']
dataset = SurgPoseInferenceDataset(img_root,ann_root,input_size=(704, 512), heatmap_size=(176,128), video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [18]:
# Define constants for metric computation
SIGMAS =  torch.tensor([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079]).to(device) #np.ones(7) / 7
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)

In [19]:
from evaluation_utils import evaluate_HRNet_full_image

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root, w_m=176.0, h_m=128.0)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)

/srv/homes/onbo10/.conda/envs/HRNetEnv_v2/lib/python3.10/site-packages/ultralytics/utils/metrics.py:184: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = torch.tensor(sigma, device=kpt1.device, dtype=kpt1.dtype)  # (17, )



YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.9409
Recall:       0.9997
mAP@50:       0.9830
mAP@50-95:    0.8800


OTHER TESTS

* evaluate yolo using the ultralytics built in functions

In [14]:
# Run evaluation on the TEST set with the integrated ultralytics frame
metrics = yolo_model.val(split='test') 

# Print the specific Pose (Keypoint) results
print(f"Test Set mAP@50-95 (Pose): {metrics.pose.map:.4f}")
print(f"Test Set mAP@50 (Pose):    {metrics.pose.map50:.4f}")
print(f"Test Set Precision (Pose): {metrics.pose.mp:.4f}")
print(f"Test Set Recall (Pose):    {metrics.pose.mr:.4f}")

Ultralytics 8.4.0 🚀 Python-3.10.19 torch-2.0.1+cu118 CUDA:0 (NVIDIA A40, 45634MiB)
YOLOv8x-pose summary (fused): 121 layers, 69,454,914 parameters, 0 gradients, 263.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4673.3±2937.3 MB/s, size: 467.0 KB)
val: Scanning /srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/labels/test.cache... 2004 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2004/2004 240.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 126/126 4.3it/s 29.5s0.2s
                   all       2004       4008      0.991      0.992      0.991      0.982      0.993      0.994      0.992      0.991
Speed: 0.3ms preprocess, 10.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /srv/homes/onbo10/thesis_Ons/Evaluation/runs/pose/val
Test Set mAP@50-95 (Pose): 0.9911
Test Set mAP@50 (Pose):    0.9922
Test Set Precision (P

* Evaluate yolo on the full test set (including corrupted GT bboxes instances)

In [ ]:
from evaluation_dataset import YoloPoseDataset
from ultralytics import YOLO
from evaluation_utils import process_batch
from ultralytics.utils.metrics import ap_per_class

device= get_device()
IOU_THRESHOLDS = torch.linspace(0.5, 0.95, 10).to(device)
SIGMAS_YOLO= torch.tensor([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079]).to(device) #
yolo_model = YOLO('/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/Yolo/runs/pose/yolov8pose_surgpose2/weights/last.pt')
dataset = YoloPoseDataset(img_dir='/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/images/test', label_dir='/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/labels/test')
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

yolo_stats = []

for batch in dataloader:
   
    img_list = [img.numpy().astype(np.uint8) for img in batch["img"].permute(0, 2, 3, 1)]
    results = yolo_model(img_list, verbose=False)
    
    h_orig, w_orig = batch["h_w_orig"]
    gt_kpts = batch["gt_kpts"][0].to(device)   # [N, 7, 3]
    gt_bboxes = batch["gt_bboxes"][0].to(device) # [N, 4]

    for r in results:
       
        pk = r.keypoints.data.clone() # [M, 7, 3]
        if pk.shape[0] > 0:
            pk[:, :, 0] = (pk[:, :, 0] / 640) * w_orig.to(device)
            pk[:, :, 1] = (pk[:, :, 1] / 640) * h_orig.to(device)
            obj_conf = r.boxes.conf.clone()
        else:
            obj_conf = torch.zeros(0).to(device)

        gt_areas = (gt_bboxes[:, 2] - gt_bboxes[:, 0]) * (gt_bboxes[:, 3] - gt_bboxes[:, 1]) * 0.53

    
        tp_matrix = process_batch(pk, gt_kpts, gt_areas, SIGMAS_YOLO, obj_conf, IOU_THRESHOLDS)
        
        yolo_stats.append((tp_matrix.cpu(), obj_conf.cpu(), torch.zeros(len(obj_conf)), torch.zeros(len(gt_kpts))))


tp, conf, pcls, gcls = [torch.cat(x, 0).numpy() for x in zip(*yolo_stats)]
results_custom = ap_per_class(tp, conf, pcls, gcls)

tp_res, fp_res, p, r, f1, ap, unique_classes = results_custom[0], results_custom[1], results_custom[2], results_custom[3], results_custom[4], results_custom[5], results_custom[6]
map50_95 = ap.mean()
map50 = ap[:, 0].mean() 
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Images: {len(yolo_stats)}")
print(f"Precision:    {p.mean():.4f}")
print(f"Recall:       {r.mean():.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


YOLOv8x-pose TEST EVALUATION RESULTS 
Total Images: 2004
Precision:    0.9839
Recall:       0.9888
mAP@50:       0.9910
mAP@50-95:    0.9848
